In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urljoin, quote_plus

URL_BASE = "https://books.toscrape.com/"
OPEN_LIBRARY_API = "https://openlibrary.org/api/books?bibkeys=ISBN:9780140328721&format=json&jscmd=data"

ESTRELLAS = {
    'One': 1, 
    'Two': 2, 
    'Three': 3, 
    'Four': 4, 
    'Five': 5
}

def crear_sesion():
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'})
    return session

def obtener_soup(url, session):
    try:
        respuesta = session.get(url)    
        respuesta.raise_for_status()
        soup = BeautifulSoup(respuesta.text, 'lxml')
        return soup
    
    except Exception as error:
        print(f"Error en {url}: {error}")
        return None
    
def extraer_datos_libro(libros, categoria_nombre):
    titulo = libros.find('h3').a['title']
    precio_text = libros.find("p", class_="price_color").text
    precio_simb = precio_text.replace("£", "").replace('Â', '').strip()
    precio = float(precio_simb)
    rating = libros.find("p", class_="star-rating").get("class")
    autor = obtener_autor(titulo)

    return {
        "titulo": titulo,
        "precio": precio,
        "categoria": categoria_nombre,
        "autor": autor,
        "rating": rating[1]
    }

def obtener_links_categorias(session):
    soup = obtener_soup(URL_BASE, session)
    lista_categoria = []

    panel_lateral = soup.find("div", class_="side_categories")
    enlaces = panel_lateral.find_all("a")

    for enlace in enlaces[1:]:
        nombre_cat = enlace.text.strip()
        ruta_relativa = enlace["href"]
        link_completo = URL_BASE + ruta_relativa

        lista_categoria.append({
            "nombre": nombre_cat,
            "url": link_completo
        })

    return lista_categoria

def obtener_autor(titulo):
    titulo_codificado = quote_plus(titulo)
    
    # Usamos la API de búsqueda (search.json), pidiendo solo el campo author_name
    url = f"https://openlibrary.org/search.json?title={titulo_codificado}&fields=author_name&limit=1"
    
    try:
        # 2. Hacemos la petición
        # IMPORTANTE: Definimos un timeout de 5 seg para que no se cuelgue si la API tarda
        respuesta = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=5)
        
        if respuesta.status_code == 200:
            datos = respuesta.json()
            
            # 3. Verificamos si hay resultados ("numFound" > 0)
            if datos.get("numFound", 0) > 0:
                docs = datos.get("docs", [])
                if docs:
                    # Extraemos la lista de autores del primer libro encontrado
                    lista_autores = docs[0].get("author_name", [])
                    if lista_autores:
                        return lista_autores[0] # Devolvemos solo el primer nombre
                        
    except Exception as e:
        # Si falla la conexión, solo imprimimos un aviso y seguimos
        print(f"⚠️ No se pudo obtener autor para '{titulo}': {e}")
        
    # 4. Valor por defecto si no se encuentra nada
    return "Autor Desconocido"    

def ejecutar_scraping():
    print("Scraping")

    mi_session = crear_sesion()
    lista_libros = []

    categorias = obtener_links_categorias(mi_session)

    print(f"Se encontraron {len(categorias)} categorias.")

    for cat in categorias:
        nombre = cat["nombre"]
        url_actual = cat["url"]
        print(f"Procesando categoria: {nombre}...")

        while True:
            soup = obtener_soup(url_actual, mi_session)
            if not soup:
                break

            libros = soup.find_all("article", class_="product_pod")

            for libro in libros:
                datos = extraer_datos_libro(libro, nombre)
                lista_libros.append(datos)

            boton_next = soup.find("li", class_="next")

            if boton_next:
                enlace_next = boton_next.a['href']
                url_actual = urljoin(url_actual, enlace_next)
            else:
                break

    return lista_libros

if __name__ == "__main__":
    test = ejecutar_scraping()
    for libro in test:
        print("Datos Libros")
        for llave, valor in libro.items():
            print(f"{llave}: {valor} ")
        print("\n")


Scraping
Se encontraron 50 categorias.
Procesando categoria: Travel...


ValueError: could not convert string to float: 'Â45.17'